# Seminário — Riqueza de peixes x profundidade nas águas brasileiras

Este notebook realiza uma análise descritiva dos registros de ocorrência de peixes localizados no **Mar Territorial e na Zona Econômica Exclusiva (ZEE) do Brasil**.

## Pergunta

> Como a riqueza de espécies de peixes varia ao longo do gradiente de profundidade nas águas brasileiras?

A base mundial é utilizada apenas como fonte original. Após a limpeza inicial, todas as tabelas e gráficos ecológicos são calculados exclusivamente para o recorte brasileiro.


## Célula 1 — Importar bibliotecas

Execute esta célula primeiro.

In [ ]:
from pathlib import Path
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

print("Bibliotecas carregadas com sucesso.")

## Célula 2 — Fazer upload do arquivo ZIP

No Colab, execute a célula abaixo e envie o arquivo:

`FinalOccurrenceDataset.zip`

In [ ]:
from google.colab import files

uploaded = files.upload()

print("Arquivos enviados:")
for nome in uploaded.keys():
    print(nome)

## Célula 3 — Configurações gerais

Aqui definimos o nome do ZIP, as pastas de saída e os limites usados na análise.

In [ ]:
ARQUIVO_ZIP = Path("FinalOccurrenceDataset.zip")

PASTA_RESULTADOS = Path("resultados")
PASTA_TABELAS = PASTA_RESULTADOS / "tabelas"
PASTA_GRAFICOS = PASTA_RESULTADOS / "graficos"

PASTA_TABELAS.mkdir(parents=True, exist_ok=True)
PASTA_GRAFICOS.mkdir(parents=True, exist_ok=True)

# Limite máximo plausível para profundidade oceânica.
PROFUNDIDADE_MAXIMA = 11000

# Intervalos de profundidade para análise principal.
LARGURA_INTERVALO = 100
LIMITE_GRAFICOS = 2000

COLUNAS_ORIGINAIS = [
    "scientificName",
    "decimalLatitude",
    "decimalLongitude",
    "coordinateUncertaintyInMeters",
    "year",
    "month",
    "day",
    "depth",
    "depthAccuracy",
    "basisOfRecord",
    "issue",
    "source",
]

COLUNAS_ANALISE = [
    "scientificName",
    "decimalLatitude",
    "decimalLongitude",
    "year",
    "depth",
    "basisOfRecord",
    "source",
]

LIMITES_FAIXAS = [0, 50, 200, 1000, 2000, 4000, 6000, 11000]

ROTULOS_FAIXAS = [
    "0–50 m",
    "50–200 m",
    "200–1.000 m",
    "1.000–2.000 m",
    "2.000–4.000 m",
    "4.000–6.000 m",
    "6.000–11.000 m",
]

print("Configurações definidas.")

## Célula 4 — Funções auxiliares

Estas funções padronizam colunas, salvam gráficos e fazem a análise complementar de riqueza padronizada.

In [ ]:
def normalizar_colunas(df):
    """Corrige pequenas diferenças observadas nos CSVs."""

    df = df.copy()

    df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed:")]

    df = df.rename(
        columns={
            "coordinate": "coordinateUncertaintyInMeters",
            "depthAccur": "depthAccuracy",
            "basisOfRec": "basisOfRecord",
        }
    )

    for coluna in COLUNAS_ORIGINAIS:
        if coluna not in df.columns:
            df[coluna] = np.nan

    return df[COLUNAS_ORIGINAIS]


def salvar_grafico(nome):
    plt.tight_layout()
    plt.savefig(PASTA_GRAFICOS / nome, dpi=300, bbox_inches="tight")
    plt.show()


def riqueza_padronizada(df, n_amostra=1000, repeticoes=50, semente=42):
    """
    Sorteia o mesmo número de registros em cada faixa de profundidade
    e calcula a riqueza média observada.
    """

    rng = np.random.default_rng(semente)
    saida = []

    for faixa, grupo in df.groupby("faixa_100m", observed=True):
        n = len(grupo)

        if n < n_amostra:
            continue

        especies = grupo["scientificName"].astype(str).to_numpy()
        riquezas = []

        for _ in range(repeticoes):
            indices = rng.choice(n, size=n_amostra, replace=False)
            riquezas.append(len(np.unique(especies[indices])))

        saida.append(
            {
                "faixa_100m": str(faixa),
                "profundidade_central_m": float(grupo["profundidade_central_m"].iloc[0]),
                "n_registros": n,
                "riqueza_media_em_1000_registros": np.mean(riquezas),
                "desvio_padrao": np.std(riquezas, ddof=1),
            }
        )

    return pd.DataFrame(saida)


print("Funções criadas.")

## Célula 5 — Ler os 365 CSVs, remover duplicatas e limpar profundidade

Esta é a célula mais demorada. Ela processa todos os arquivos do ZIP.

In [ ]:
if not ARQUIVO_ZIP.exists():
    raise FileNotFoundError("O arquivo FinalOccurrenceDataset.zip não foi encontrado.")

n_bruto = 0
n_duplicatas = 0

ausentes_acumulados = pd.Series(0, index=COLUNAS_ORIGINAIS, dtype="int64")

partes_validas = []
linhas_por_arquivo = []

with zipfile.ZipFile(ARQUIVO_ZIP, "r") as z:
    arquivos_csv = [
        nome
        for nome in z.namelist()
        if nome.lower().endswith(".csv") and "__MACOSX" not in nome
    ]

    print(f"Arquivos CSV encontrados: {len(arquivos_csv)}")

    for i, nome in enumerate(arquivos_csv, start=1):
        with z.open(nome) as arquivo:
            df = pd.read_csv(arquivo, low_memory=False)

        df = normalizar_colunas(df)

        n_arquivo = len(df)
        n_bruto += n_arquivo
        ausentes_acumulados += df.isna().sum()

        numericas = [
            "decimalLatitude",
            "decimalLongitude",
            "coordinateUncertaintyInMeters",
            "year",
            "month",
            "day",
            "depth",
            "depthAccuracy",
        ]

        for coluna in numericas:
            df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

        for coluna in ["scientificName", "basisOfRecord", "source"]:
            df[coluna] = df[coluna].astype("string").str.strip()

        mascara_dup = df.duplicated(subset=COLUNAS_ORIGINAIS)
        duplicatas_arquivo = int(mascara_dup.sum())
        n_duplicatas += duplicatas_arquivo

        df = df.loc[~mascara_dup].copy()

        linhas_por_arquivo.append(
            {
                "arquivo": Path(nome).name,
                "registros_brutos": n_arquivo,
                "duplicatas_exatas": duplicatas_arquivo,
                "registros_unicos": len(df),
            }
        )

        df = df[COLUNAS_ANALISE]

        df_valido = df[
            df["scientificName"].notna()
            & df["depth"].notna()
            & (df["depth"] >= 0)
            & (df["depth"] <= PROFUNDIDADE_MAXIMA)
        ].copy()

        partes_validas.append(df_valido)

        if i % 25 == 0 or i == len(arquivos_csv):
            print(f"Processados {i}/{len(arquivos_csv)} arquivos")

dados = pd.concat(partes_validas, ignore_index=True)
del partes_validas

dados["source"] = dados["source"].str.upper()

n_unicos = n_bruto - n_duplicatas
n_validos = len(dados)
n_especies_brutas = len(arquivos_csv)
n_especies_validas = dados["scientificName"].nunique()

print("\nResumo inicial:")
print(f"Registros brutos: {n_bruto:,}")
print(f"Duplicatas exatas: {n_duplicatas:,}")
print(f"Registros únicos: {n_unicos:,}")
print(f"Registros válidos para análise: {n_validos:,}")
print(f"Espécies com profundidade válida: {n_especies_validas:,}")

## Célula 6 — Recorte espacial: águas brasileiras

A partir daqui, a análise deixa de usar os registros globais e mantém somente os pontos localizados no **Mar Territorial + Zona Econômica Exclusiva (ZEE) do Brasil**.

Os limites são obtidos diretamente do serviço geográfico oficial disponibilizado pelo IBAMA, com dados da **Marinha do Brasil**, em SIRGAS 2000.

Usamos Mar Territorial + ZEE porque usar apenas a ZEE excluiria a faixa costeira até 12 milhas náuticas, justamente uma região importante para o tema do trabalho.

In [ ]:
# Instala/atualiza as bibliotecas espaciais necessárias no Colab.
!pip -q install geopandas shapely pyogrio

import geopandas as gpd
from shapely.geometry import Point

# Serviço oficial IBAMA / Marinha do Brasil.
# Camada 0 = Mar Territorial
# Camada 2 = Zona Econômica Exclusiva (ZEE)
BASE = "https://pamgia.ibama.gov.br/server/rest/services/01_Publicacoes_Bases/lim_amazonia_azul/MapServer"

URL_MT = BASE + "/0/query?where=1%3D1&outFields=*&returnGeometry=true&f=geojson"
URL_ZEE = BASE + "/2/query?where=1%3D1&outFields=*&returnGeometry=true&f=geojson"

mar_territorial = gpd.read_file(URL_MT).to_crs("EPSG:4674")
zee = gpd.read_file(URL_ZEE).to_crs("EPSG:4674")

# Une os polígonos das duas áreas marítimas brasileiras.
area_marinha_brasil = gpd.GeoDataFrame(
    geometry=[mar_territorial.geometry.union_all().union(zee.geometry.union_all())],
    crs="EPSG:4674"
)

# Guarda uma cópia da base mundial apenas para informar o tamanho da base original.
dados_global = dados.copy()

# Remove registros sem coordenadas antes da transformação espacial.
dados_coord = dados_global[
    dados_global["decimalLatitude"].notna()
    & dados_global["decimalLongitude"].notna()
    & dados_global["decimalLatitude"].between(-90, 90)
    & dados_global["decimalLongitude"].between(-180, 180)
].copy()

# Converte os registros em pontos geográficos.
pontos = gpd.GeoDataFrame(
    dados_coord,
    geometry=gpd.points_from_xy(
        dados_coord["decimalLongitude"],
        dados_coord["decimalLatitude"]
    ),
    crs="EPSG:4326"
).to_crs("EPSG:4674")

# Interseção espacial: mantém somente pontos dentro do Mar Territorial ou da ZEE.
pontos_brasil = gpd.sjoin(
    pontos,
    area_marinha_brasil,
    how="inner",
    predicate="within"
)

# A partir desta linha, 'dados' significa SOMENTE o recorte brasileiro.
dados = pd.DataFrame(
    pontos_brasil.drop(columns=["geometry", "index_right"], errors="ignore")
).reset_index(drop=True)

n_registros_brasil = len(dados)
n_especies_brasil = dados["scientificName"].nunique()

print("RECORTE BRASILEIRO CONCLUÍDO")
print("=" * 60)
print(f"Registros válidos globais antes do recorte: {len(dados_global):,}")
print(f"Registros nas águas brasileiras: {n_registros_brasil:,}")
print(f"Espécies nas águas brasileiras: {n_especies_brasil:,}")
print(f"Profundidade mínima: {dados['depth'].min():.1f} m")
print(f"Profundidade mediana: {dados['depth'].median():.1f} m")
print(f"Profundidade máxima: {dados['depth'].max():.1f} m")

## Célula 7 — Conferência visual do recorte brasileiro

Este mapa serve para confirmar se os pontos selecionados realmente estão nas águas brasileiras e pode ser usado no slide de área de estudo.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 9))

area_marinha_brasil.boundary.plot(ax=ax, linewidth=1)

# Amostra de pontos apenas para o mapa ficar leve e legível.
amostra_mapa = pontos_brasil.sample(
    n=min(5000, len(pontos_brasil)),
    random_state=42
)

amostra_mapa.plot(ax=ax, markersize=3, alpha=0.35)

ax.set_title("Registros de peixes selecionados nas águas brasileiras")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.tight_layout()
plt.savefig(PASTA_GRAFICOS / "00_recorte_brasil.png", dpi=300, bbox_inches="tight")
plt.show()

## Célula 6 — Tabela de dados ausentes

Essa tabela mostra quais variáveis tinham mais dados faltantes na base original.

In [ ]:
ausentes = ausentes_acumulados.to_frame(name="n_ausentes")
ausentes["percentual_ausente"] = 100 * ausentes["n_ausentes"] / n_bruto

ausentes.to_csv(PASTA_TABELAS / "01_dados_ausentes.csv", encoding="utf-8-sig")

ausentes.sort_values("percentual_ausente", ascending=False)

## Célula 7 — Gráfico de dados ausentes

Gráfico útil para o slide de limpeza e qualidade dos dados.

In [ ]:
ausentes_plot = ausentes.sort_values("percentual_ausente", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(ausentes_plot.index, ausentes_plot["percentual_ausente"])
plt.xlabel("Dados ausentes (%)")
plt.ylabel("Variável")
plt.title("Percentual de dados ausentes por variável")
salvar_grafico("01_dados_ausentes.png")

## Célula 8 — Resumo da limpeza

Tabela que resume quantos registros foram mantidos após a limpeza.

In [ ]:
resumo_limpeza = pd.DataFrame(
    {
        "indicador": [
            "Arquivos CSV",
            "Registros brutos",
            "Duplicatas exatas",
            "Registros únicos",
            "Registros válidos para análise",
            "Espécies na base",
            "Espécies com profundidade válida",
        ],
        "valor": [
            len(arquivos_csv),
            n_bruto,
            n_duplicatas,
            n_unicos,
            n_validos,
            n_especies_brutas,
            n_especies_validas,
        ],
    }
)

resumo_limpeza.to_csv(PASTA_TABELAS / "02_resumo_limpeza.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(linhas_por_arquivo).to_csv(PASTA_TABELAS / "03_registros_por_arquivo.csv", index=False, encoding="utf-8-sig")

resumo_limpeza

## Célula 9 — Tabela descritiva

Aqui são calculadas medidas como média, mediana, desvio-padrão, mínimo, máximo e quartis.

In [ ]:
descricao = dados[["depth", "decimalLatitude", "decimalLongitude", "year"]].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

descricao = descricao.rename(
    columns={
        "count": "n",
        "mean": "media",
        "std": "desvio_padrao",
        "min": "minimo",
        "1%": "p01",
        "5%": "p05",
        "25%": "q1",
        "50%": "mediana",
        "75%": "q3",
        "95%": "p95",
        "99%": "p99",
        "max": "maximo",
    }
)

descricao.to_csv(PASTA_TABELAS / "04_tabela_descritiva.csv", encoding="utf-8-sig")

descricao

## Célula 10 — Histograma da profundidade

Mostra onde os registros estão concentrados. Usamos 0 a 2.000 m para o gráfico ficar legível.

In [ ]:
dados_2000 = dados[dados["depth"] <= LIMITE_GRAFICOS].copy()

plt.figure(figsize=(10, 6))
plt.hist(dados_2000["depth"], bins=40)
plt.xlabel("Profundidade (m)")
plt.ylabel("Número de registros")
plt.title("Distribuição dos registros por profundidade (0–2.000 m)")
salvar_grafico("02_histograma_profundidade.png")

## Célula 11 — Fontes dos registros

Identifica a origem dos dados, como OBIS e GBIF.

In [ ]:
fontes = (
    dados["source"]
    .fillna("NÃO INFORMADA")
    .value_counts()
    .rename_axis("fonte")
    .reset_index(name="n_registros")
)

fontes["percentual"] = 100 * fontes["n_registros"] / fontes["n_registros"].sum()
fontes.to_csv(PASTA_TABELAS / "05_fontes.csv", index=False, encoding="utf-8-sig")

fontes.head(10)

## Célula 12 — Gráfico das fontes dos registros

In [ ]:
fontes_plot = fontes.head(8).sort_values("n_registros", ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(fontes_plot["fonte"], fontes_plot["n_registros"])
plt.xlabel("Número de registros válidos")
plt.ylabel("Fonte")
plt.title("Principais fontes dos registros")
salvar_grafico("03_fontes_dados.png")

## Célula 13 — Espécies mais registradas

Mostra que algumas espécies têm muito mais registros que outras.

In [ ]:
top_especies = (
    dados["scientificName"]
    .value_counts()
    .head(10)
    .rename_axis("especie")
    .reset_index(name="n_registros")
)

top_especies.to_csv(PASTA_TABELAS / "06_top_10_especies.csv", index=False, encoding="utf-8-sig")

top_especies

## Célula 14 — Gráfico das espécies mais registradas

In [ ]:
top_plot = top_especies.sort_values("n_registros", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(top_plot["especie"], top_plot["n_registros"])
plt.xlabel("Número de registros válidos")
plt.ylabel("Espécie")
plt.title("Dez espécies com maior número de registros")
salvar_grafico("04_top_10_especies.png")

## Célula 15 — Riqueza por faixas amplas de profundidade

Aqui calculamos quantas espécies diferentes aparecem em cada faixa de profundidade.

In [ ]:
dados["faixa_ampla"] = pd.cut(
    dados["depth"],
    bins=LIMITES_FAIXAS,
    labels=ROTULOS_FAIXAS,
    right=False,
    include_lowest=True,
)

faixas_amplas = (
    dados
    .groupby("faixa_ampla", observed=False)
    .agg(
        n_registros=("scientificName", "size"),
        riqueza_especies=("scientificName", "nunique"),
    )
    .reset_index()
)

faixas_amplas.to_csv(PASTA_TABELAS / "07_riqueza_por_faixas_amplas.csv", index=False, encoding="utf-8-sig")

faixas_amplas

## Célula 16 — Gráfico de riqueza por faixas amplas

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(faixas_amplas["faixa_ampla"].astype(str), faixas_amplas["riqueza_especies"])
plt.xlabel("Faixa de profundidade")
plt.ylabel("Número de espécies")
plt.title("Riqueza de espécies por faixas amplas de profundidade")
plt.xticks(rotation=35, ha="right")
salvar_grafico("05_riqueza_faixas_amplas.png")

## Célula 17 — Riqueza por intervalos iguais de 100 m

Esta é a análise principal do trabalho, porque compara faixas de mesma largura.

In [ ]:
bins_100 = np.arange(0, PROFUNDIDADE_MAXIMA + LARGURA_INTERVALO, LARGURA_INTERVALO)

dados["faixa_100m"] = pd.cut(
    dados["depth"],
    bins=bins_100,
    right=False,
    include_lowest=True,
)

dados["profundidade_central_m"] = (
    dados["faixa_100m"]
    .apply(lambda x: x.mid if pd.notna(x) else np.nan)
    .astype(float)
)

riqueza_100m = (
    dados
    .groupby(["faixa_100m", "profundidade_central_m"], observed=True)
    .agg(
        n_registros=("scientificName", "size"),
        riqueza_especies=("scientificName", "nunique"),
    )
    .reset_index()
)

riqueza_100m["faixa_100m"] = riqueza_100m["faixa_100m"].astype(str)

riqueza_100m.to_csv(PASTA_TABELAS / "08_riqueza_por_100m.csv", index=False, encoding="utf-8-sig")

principal = riqueza_100m[riqueza_100m["profundidade_central_m"] <= LIMITE_GRAFICOS].copy()

principal.head(20)

## Célula 18 — Gráfico principal: riqueza x profundidade

Este é o gráfico mais importante para os slides.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(principal["profundidade_central_m"], principal["riqueza_especies"], marker="o")
plt.xlabel("Profundidade (m)")
plt.ylabel("Riqueza de espécies")
plt.title("Riqueza de espécies ao longo da profundidade oceânica")
plt.grid(alpha=0.25)
salvar_grafico("06_riqueza_por_100m.png")

## Célula 19 — Esforço amostral x profundidade

Este gráfico mostra que o número de registros cai bastante com a profundidade.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(principal["profundidade_central_m"], principal["n_registros"], marker="o")
plt.yscale("log")
plt.xlabel("Profundidade (m)")
plt.ylabel("Número de registros (escala log)")
plt.title("Número de registros ao longo da profundidade")
plt.grid(alpha=0.25)
salvar_grafico("07_registros_por_100m.png")

## Célula 20 — Riqueza observada x número de registros

Ajuda a discutir o efeito do esforço amostral.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(principal["n_registros"], principal["riqueza_especies"])

for _, linha in principal.iterrows():
    inicio = int(linha["profundidade_central_m"] - 50)
    fim = int(linha["profundidade_central_m"] + 50)
    plt.annotate(
        f"{inicio}–{fim}",
        (linha["n_registros"], linha["riqueza_especies"]),
        fontsize=7,
        alpha=0.75,
    )

plt.xscale("log")
plt.xlabel("Número de registros na faixa (escala log)")
plt.ylabel("Riqueza de espécies")
plt.title("Riqueza observada e esforço amostral")
plt.grid(alpha=0.25)
salvar_grafico("08_riqueza_vs_registros.png")

## Célula 21 — Análise complementar: riqueza padronizada por 1.000 registros

Esta análise sorteia 1.000 registros de cada faixa e calcula a riqueza média. Ela ajuda a discutir o viés causado pelo número desigual de registros por profundidade.

In [ ]:
riqueza_1000 = riqueza_padronizada(
    dados[dados["profundidade_central_m"] <= LIMITE_GRAFICOS],
    n_amostra=1000,
    repeticoes=50,
    semente=42,
)

riqueza_1000.to_csv(PASTA_TABELAS / "09_riqueza_padronizada_1000_registros.csv", index=False, encoding="utf-8-sig")

riqueza_1000.head(20)

## Célula 22 — Gráfico da riqueza padronizada

Este gráfico é excelente para a discussão final.

In [ ]:
plt.figure(figsize=(10, 6))
plt.errorbar(
    riqueza_1000["profundidade_central_m"],
    riqueza_1000["riqueza_media_em_1000_registros"],
    yerr=riqueza_1000["desvio_padrao"],
    marker="o",
    capsize=3,
)
plt.xlabel("Profundidade (m)")
plt.ylabel("Riqueza média em 1.000 registros")
plt.title("Riqueza com esforço amostral padronizado")
plt.grid(alpha=0.25)
salvar_grafico("09_riqueza_padronizada.png")

## Célula 23 — Distribuição temporal dos registros

Mostra em quais anos há mais registros na base.

In [ ]:
anos = dados["year"].dropna()
anos = anos[(anos >= 1700) & (anos <= 2100)]

registros_ano = (
    anos.astype(int)
    .value_counts()
    .sort_index()
    .rename_axis("ano")
    .reset_index(name="n_registros")
)

registros_ano.to_csv(PASTA_TABELAS / "10_registros_por_ano.csv", index=False, encoding="utf-8-sig")

plt.figure(figsize=(10, 5))
plt.plot(registros_ano["ano"], registros_ano["n_registros"])
plt.xlabel("Ano")
plt.ylabel("Número de registros")
plt.title("Distribuição temporal dos registros")
salvar_grafico("10_registros_por_ano.png")

## Célula 24 — Resumo final para apresentação

Esta célula imprime os principais números que podem ser usados nos slides.

In [ ]:
print("RESUMO FINAL — ÁGUAS BRASILEIRAS")
print("=" * 60)
print(f"Arquivos CSV na base original: {len(arquivos_csv):,}")
print(f"Registros brutos na base original: {n_bruto:,}")
print(f"Registros válidos após o recorte brasileiro: {len(dados):,}")
print(f"Espécies no recorte brasileiro: {dados['scientificName'].nunique():,}")
print(f"Profundidade mediana no Brasil: {dados['depth'].median():.1f} m")
print(f"Profundidade máxima no Brasil: {dados['depth'].max():.1f} m")
print("=" * 60)

## Célula 25 — Compactar resultados para baixar

Execute esta célula para baixar todos os gráficos e tabelas gerados.

In [ ]:
import shutil

arquivo_zip_resultados = shutil.make_archive("resultados_depth_matters", "zip", PASTA_RESULTADOS)

files.download(arquivo_zip_resultados)